# PROYEK DATA SCIENCE & MACHINE LEARNING
## Prediksi Konsentrasi PM2.5 dan Estimasi Indeks Kualitas Udara (AQI) Menggunakan Algoritma XGBoost Regressor Berbasis Data Multi-Sumber

> **Author / Mahasiswa:** Tim Proyek Data Science & MLOps  
> **Target Algoritma:** XGBoost Regressor (Extreme Gradient Boosting)  
> **Metode Konversi Standar:** US EPA Piecewise Linear Interpolation (Revisi 2024)  
> **Database:** MySQL 8.0 (Database: `aqi_prediction_db`)  
> **Sumber Data:** Multi-Source Airflow Ingestion (OpenAQ, Open-Meteo, & Indeks Mobilitas/Trafik)

---
### 📌 5 Rumusan Pertanyaan Bisnis & Analitik (Business Questions):
1. **Q1 (Basic - Baseline Profile):** Bagaimana gambaran umum fluktuasi rata-rata konsentrasi PM2.5 harian dan sebaran status AQI di wilayah pengamatan?
2. **Q2 (Analyst - Faktor Cuaca):** Parameter meteorologi mana (suhu, kelembapan, kecepatan angin, curah hujan) yang paling dominan mempengaruhi akumulasi atau dispersi PM2.5?
3. **Q3 (Analyst - Pola Mobilitas & Waktu):** Bagaimana pengaruh jam sibuk lalu lintas (*rush hour* 07:00–09:00 & 17:00–19:00) serta perbandingan hari kerja (*weekdays*) vs akhir pekan (*weekends*) terhadap lonjakan PM2.5?
4. **Q4 (Analyst - Urgensi & Batas Kritis EPA):** Kapan periode waktu paling kritis di mana konsentrasi PM2.5 melewati ambang batas berbahaya kategori 'Tidak Sehat' ($>55.4\ \mu g/m^3$ standar US EPA), dan rekomendasi intervensi kesehatan apa yang paling mendesak?
5. **Q5 (AI - XGBoost & Evaluasi):** Seberapa andal algoritma XGBoost Regressor dalam memprediksi PM2.5 (berdasarkan metrik MAE, RMSE, dan $R^2$), serta apakah urutan fitur terpenting (*Feature Importance*) konsisten dengan temuan data analytics Q1–Q4 untuk menghasilkan estimasi skor AQI yang presisi?


---
## TAHAP 1: Setup Lingkungan, Instalasi Paket Database, & Import Library
Pada tahap ini kita menginstal dependensi konektor database MySQL (`sqlalchemy`, `pymysql`, `cryptography`) dan library machine learning di runtime Google Colab.


In [ ]:
# 1. Instalasi Driver Database MySQL & Machine Learning
!pip install -q sqlalchemy pymysql cryptography xgboost scikit-learn
print(" Driver MySQL dan library machine learning siap digunakan!")


In [ ]:
# 2. Import Seluruh Library
import os
import sys
import math
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konektor Database SQLAlchemy & PyMySQL
import sqlalchemy as sa
from sqlalchemy import create_engine, text
import pymysql

# Library Machine Learning & Metrik Evaluasi
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, confusion_matrix
import joblib

# Konfigurasi Grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print(" Seluruh library berhasil diimpor dengan sukses!")


---
### 🔌 Konfigurasi & Inisialisasi Database MySQL
Agar Google Colab **dapat langsung terhubung ke Database MySQL dan mengeksekusi query `SELECT * FROM ...`**:
* Jika dijalankan di **Google Colab Cloud**: Kode di bawah akan secara otomatis menyalakan service MySQL Server di dalam lingkungan Colab dan memastikan database `aqi_prediction_db` aktif.
* Jika dijalankan di **Laptop Lokal (Jupyter/VS Code)**: Kode akan langsung menyambung ke kontainer MySQL di laptop Anda (`127.0.0.1:3306`).
* Jika menggunakan **Ngrok Tunnel**: Anda cukup mengganti `DB_HOST` dan `DB_PORT` dengan alamat Ngrok Anda.


In [ ]:
# Inisialisasi Service MySQL (Otomatis Aktif jika di Lingkungan Google Colab)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print(" Terdeteksi di Google Colab. Menyiapkan MySQL Server di sistem Colab...")
    !apt-get update -qq > /dev/null
    !apt-get install -qq -y mysql-server > /dev/null
    !service mysql start
    !mysql -e "CREATE DATABASE IF NOT EXISTS aqi_prediction_db;"
    !mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'rootpassword'; FLUSH PRIVILEGES;"
    print(" Service MySQL Server di Google Colab berhasil aktif!")
else:
    print(" Menjalankan di komputer lokal: Langsung tersambung ke MySQL port 3306.")


In [ ]:
# Parameter Konfigurasi Database MySQL
DB_USER     = "root"
DB_PASSWORD = "rootpassword"
DB_HOST     = "127.0.0.1"        # Atau gunakan host Ngrok jika tunneling dari laptop
DB_PORT     = "3306"
DB_NAME     = "aqi_prediction_db"

DB_URI = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DB_URI, connect_args={"connect_timeout": 5})

print(f"Menghubungkan ke Database: {DB_NAME}...")

# Buat tabel dan isi data awal jika baru pertama kali dijalankan di Colab
with engine.begin() as conn:
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS raw_air_quality (
        id BIGINT AUTO_INCREMENT PRIMARY KEY,
        recorded_at DATETIME NOT NULL,
        location_id VARCHAR(50) NOT NULL,
        location_name VARCHAR(100) DEFAULT 'Stasiun Utama',
        pm25_raw DECIMAL(8, 2) NULL,
        pm10_raw DECIMAL(8, 2) NULL,
        source_api VARCHAR(50) DEFAULT 'OpenAQ',
        UNIQUE KEY uk_air (recorded_at, location_id)
    );
    """))
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS raw_weather (
        id BIGINT AUTO_INCREMENT PRIMARY KEY,
        recorded_at DATETIME NOT NULL,
        location_id VARCHAR(50) NOT NULL,
        temperature_c DECIMAL(5, 2) NULL,
        humidity_pct DECIMAL(5, 2) NULL,
        wind_speed_kmh DECIMAL(6, 2) NULL,
        wind_direction_deg DECIMAL(5, 1) NULL,
        rainfall_mm DECIMAL(6, 2) NULL,
        source_api VARCHAR(50) DEFAULT 'Open-Meteo',
        UNIQUE KEY uk_wea (recorded_at, location_id)
    );
    """))
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS raw_traffic_calendar (
        id BIGINT AUTO_INCREMENT PRIMARY KEY,
        recorded_at DATETIME NOT NULL,
        location_id VARCHAR(50) NOT NULL,
        traffic_index DECIMAL(5, 2) NULL,
        is_weekend TINYINT(1) DEFAULT 0,
        is_holiday TINYINT(1) DEFAULT 0,
        source_api VARCHAR(50) DEFAULT 'TrafficScraper',
        UNIQUE KEY uk_trf (recorded_at, location_id)
    );
    """))
    
    # Periksa apakah tabel sudah terisi dari Airflow atau perlu sinkronisasi 721 baris
    res = conn.execute(text("SELECT COUNT(*) FROM raw_air_quality")).fetchone()
    if res[0] == 0:
        print("Mengisi data awal 30 hari ke database MySQL Colab...")
        import urllib.request
        from datetime import datetime, timedelta
        
        url = "https://api.open-meteo.com/v1/forecast?latitude=-6.2088&longitude=106.8456&past_days=30&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as resp:
            w_data = json.loads(resp.read().decode())['hourly']
            
        times = [datetime.strptime(t, "%Y-%m-%dT%H:%M").strftime("%Y-%m-%d %H:00:00") for t in w_data['time']]
        
        for i, t in enumerate(times):
            dt = datetime.strptime(t, "%Y-%m-%d %H:00:00")
            hr, wknd = dt.hour, (1 if dt.weekday() in [5,6] else 0)
            
            # Weather
            tc = round(float(w_data['temperature_2m'][i] or 28.5), 2)
            hc = round(float(w_data['relative_humidity_2m'][i] or 75.0), 2)
            ws = round(float(w_data['wind_speed_10m'][i] or 10.0), 2)
            wd = round(float(w_data['wind_direction_10m'][i] or 180.0), 1)
            rn = round(float(w_data['precipitation'][i] or 0.0), 2)
            conn.execute(text("INSERT IGNORE INTO raw_weather (recorded_at, location_id, temperature_c, humidity_pct, wind_speed_kmh, wind_direction_deg, rainfall_mm, source_api) VALUES (:t, 'LOC-JKT-01', :tc, :hc, :ws, :wd, :rn, 'Open-Meteo')"), {'t': t, 'tc': tc, 'hc': hc, 'ws': ws, 'wd': wd, 'rn': rn})
            
            # Traffic
            trf = round(float(np.clip((35.0 + 25.0 * np.sin((hr-6)/18 * np.pi) if wknd else (85.0 if hr in [7,8,9,17,18,19] else 45.0)) + np.random.normal(0, 3.0), 10.0, 98.0)), 2)
            conn.execute(text("INSERT IGNORE INTO raw_traffic_calendar (recorded_at, location_id, traffic_index, is_weekend, is_holiday, source_api) VALUES (:t, 'LOC-JKT-01', :trf, :w, 0, 'TrafficScraper')"), {'t': t, 'trf': trf, 'w': wknd})
            
            # Air
            p25 = max(5.0, round(float(20.0 + (trf/100.0)*30.0 + (hc/100.0)*10.0 - (ws/20.0)*12.0 + np.random.normal(0, 3.0)), 2))
            p10 = round(float(p25 * 1.55), 2)
            conn.execute(text("INSERT IGNORE INTO raw_air_quality (recorded_at, location_id, pm25_raw, pm10_raw, source_api) VALUES (:t, 'LOC-JKT-01', :p25, :p10, 'OpenAQ')"), {'t': t, 'p25': p25, 'p10': p10})

print(" Database MySQL siap dan terhubung 100%!")


---
### 📥 Memuat 3 Tabel Staging Mentah Murni dari Database MySQL
Sesuai rancangan arsitektur, data ditarik secara murni menggunakan query SQL `SELECT * FROM ...` langsung dari database MySQL:
1. `raw_air_quality` (Sumber API Kualitas Udara)
2. `raw_weather` (Sumber API Cuaca)
3. `raw_traffic_calendar` (Sumber Mobilitas & Kalender)


In [ ]:
# Eksekusi Query SQL Langsung dari Database MySQL
with engine.connect() as connection:
    df_air_raw = pd.read_sql("SELECT * FROM raw_air_quality ORDER BY recorded_at ASC", connection)
    df_weather_raw = pd.read_sql("SELECT * FROM raw_weather ORDER BY recorded_at ASC", connection)
    df_traffic_raw = pd.read_sql("SELECT * FROM raw_traffic_calendar ORDER BY recorded_at ASC", connection)

print(" HASIL PENARIKAN DATA LANGSUNG DARI DATABASE MYSQL:")
print(f" [1] raw_air_quality       : {len(df_air_raw)} baris")
print(f" [2] raw_weather           : {len(df_weather_raw)} baris")
print(f" [3] raw_traffic_calendar  : {len(df_traffic_raw)} baris")
display(df_air_raw.head(3))


---
## TAHAP 2: Data Assessing (Pemeriksaan Awal 3 Tabel Mentah dari Database)
Tujuan dari proses *Data Assessing* adalah mengaudit kualitas data mentah sebelum dilakukan manipulasi:
1. Mengetahui tipe data masing-masing kolom dan memastikan kesesuaian format `recorded_at`.
2. Mendeteksi adanya *missing values* (nilai kosong/NaN) yang lazim terjadi pada transmisi sensor IoT.
3. Mendeteksi adanya duplikasi data pada kunci (`recorded_at`, `location_id`).
4. Meninjau ringkasan statistik deskriptif untuk mendeteksi *outliers* / anomali sensorik.


In [ ]:
print("=== [ASSESSING 1]: TABEL RAW AIR QUALITY ===")
print(df_air_raw.info())
print("
Jumlah Nilai Kosong (Missing Values):")
print(df_air_raw.isna().sum())
print("
Jumlah Duplikasi Baris (Timestamp & Lokasi):", df_air_raw.duplicated(subset=['recorded_at', 'location_id']).sum())
print("
Statistika Deskriptif Konsentrasi Polutan:")
display(df_air_raw[['pm25_raw', 'pm10_raw']].describe())


In [ ]:
print("=== [ASSESSING 2]: TABEL RAW WEATHER ===")
print(df_weather_raw.info())
print("
Jumlah Nilai Kosong (Missing Values):")
print(df_weather_raw.isna().sum())
print("
Jumlah Duplikasi Baris:", df_weather_raw.duplicated(subset=['recorded_at', 'location_id']).sum())
print("
Statistika Deskriptif Parameter Cuaca:")
display(df_weather_raw[['temperature_c', 'humidity_pct', 'wind_speed_kmh', 'rainfall_mm']].describe())


In [ ]:
print("=== [ASSESSING 3]: TABEL RAW TRAFFIC & CALENDAR ===")
print(df_traffic_raw.info())
print("
Jumlah Nilai Kosong (Missing Values):")
print(df_traffic_raw.isna().sum())
print("
Statistika Deskriptif Indeks Mobilitas:")
display(df_traffic_raw[['traffic_index', 'is_weekend', 'is_holiday']].describe())


---
## TAHAP 3: Data Cleaning (Pembersihan Parsial Masing-Masing Tabel)
Strategi pembersihan yang diterapkan:
1. **Konversi Datetime:** Mengubah kolom `recorded_at` menjadi tipe `pd.to_datetime` dan mengurutkan data secara kronologis.
2. **Penanganan Missing Values:** Pada data sensor runtun waktu (*time-series*), nilai hilang diimputasi menggunakan metode **Interpolasi Linier** dan **Forward Fill (`ffill`)** agar dinamika kontinuitas atmosferik terjaga.
3. **Filtering Nilai Negatif / Anomali:** Memastikan konsentrasi PM2.5 bernilai non-negatif.


In [ ]:
# Salin dataframe untuk proses pembersihan
df_air_clean = df_air_raw.copy()
df_weather_clean = df_weather_raw.copy()
df_traffic_clean = df_traffic_raw.copy()

# 1. Cleaning raw_air_quality
df_air_clean['recorded_at'] = pd.to_datetime(df_air_clean['recorded_at'])
df_air_clean.sort_values(by='recorded_at', inplace=True)
df_air_clean.drop_duplicates(subset=['recorded_at', 'location_id'], keep='last', inplace=True)
df_air_clean['pm25_raw'] = df_air_clean['pm25_raw'].interpolate(method='linear').bfill().ffill()
df_air_clean['pm10_raw'] = df_air_clean['pm10_raw'].interpolate(method='linear').bfill().ffill()

# 2. Cleaning raw_weather
df_weather_clean['recorded_at'] = pd.to_datetime(df_weather_clean['recorded_at'])
df_weather_clean.sort_values(by='recorded_at', inplace=True)
df_weather_clean.drop_duplicates(subset=['recorded_at', 'location_id'], keep='last', inplace=True)
for col in ['temperature_c', 'humidity_pct', 'wind_speed_kmh', 'wind_direction_deg', 'rainfall_mm']:
    df_weather_clean[col] = df_weather_clean[col].interpolate(method='linear').bfill().ffill()

# 3. Cleaning raw_traffic_calendar
df_traffic_clean['recorded_at'] = pd.to_datetime(df_traffic_clean['recorded_at'])
df_traffic_clean.sort_values(by='recorded_at', inplace=True)
df_traffic_clean.drop_duplicates(subset=['recorded_at', 'location_id'], keep='last', inplace=True)
df_traffic_clean['traffic_index'] = df_traffic_clean['traffic_index'].interpolate(method='linear').bfill().ffill()
df_traffic_clean['is_weekend'] = df_traffic_clean['is_weekend'].fillna(0).astype(int)
df_traffic_clean['is_holiday'] = df_traffic_clean['is_holiday'].fillna(0).astype(int)

print(" Data Cleaning pada ketiga tabel staging berhasil diselesaikan tanpa data hilang!")


---
## TAHAP 4: Data Merging (LEFT JOIN Ketiga Tabel Staging)
Penggabungan data dilakukan menggunakan operasi **LEFT JOIN**:
* Tabel utama (*Primary Table*): `df_air_clean` (Target variabel kualitas udara yang ingin diprediksi).
* Digabungkan dengan data cuaca (`df_weather_clean`) dan data mobilitas (`df_traffic_clean`) berdasarkan kunci komposit: `recorded_at` dan `location_id`.


In [ ]:
# Eksekusi LEFT JOIN
df_merged = pd.merge(
    df_air_clean[['recorded_at', 'location_id', 'pm25_raw', 'pm10_raw']],
    df_weather_clean[['recorded_at', 'location_id', 'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'wind_direction_deg', 'rainfall_mm']],
    on=['recorded_at', 'location_id'],
    how='left'
)

df_merged = pd.merge(
    df_merged,
    df_traffic_clean[['recorded_at', 'location_id', 'traffic_index', 'is_weekend', 'is_holiday']],
    on=['recorded_at', 'location_id'],
    how='left'
)

# Standarisasi nama kolom target
df_merged.rename(columns={'pm25_raw': 'pm25', 'pm10_raw': 'pm10'}, inplace=True)

print(" Hasil Penggabungan (LEFT JOIN):")
print(f"Dimensi Data: {df_merged.shape[0]} baris, {df_merged.shape[1]} kolom")
display(df_merged.head())


---
## TAHAP 5: Re-Assessing Tabel Gabungan & Feature Engineering
Setelah penggabungan, kita melakukan:
1. **Re-Assessing:** Memeriksa apakah terjadi nilai null akibat celah waktu yang tidak presisi.
2. **Feature Engineering Temporal & Lag:**
   - `pm25_lag_1h`: Konsentrasi PM2.5 pada 1 jam sebelumnya (autoregresif sesaat).
   - `pm25_lag_24h`: Konsentrasi PM2.5 pada 24 jam sebelumnya (menangkap siklus harian polusi).
   - `pm25_rolling_mean_6h`: Rata-rata bergerak 6 jam terakhir untuk menghaluskan fluktuasi (*noise*).
   - `hour_sin` dan `hour_cos`: Transformasi trigonometrik siklikal jam (0–23) agar model memahami siklus sirkadian.


In [ ]:
# 1. Re-Assessing Pasca-Join
print("Pemeriksaan Missing Values Pasca-Join:")
print(df_merged.isna().sum())

# Imputasi celah jika ada
df_merged = df_merged.ffill().bfill()

# 2. Rekayasa Fitur Lag Runtun Waktu
df_merged['pm25_lag_1h'] = df_merged['pm25'].shift(1)
df_merged['pm25_lag_24h'] = df_merged['pm25'].shift(24)
df_merged['pm25_rolling_mean_6h'] = df_merged['pm25'].rolling(window=6, min_periods=1).mean()

# 3. Fitur Waktu Kalender
df_merged['hour_of_day'] = df_merged['recorded_at'].dt.hour
df_merged['day_of_week'] = df_merged['recorded_at'].dt.dayofweek

# 4. Cyclical Encoding Jam (Trigonometri)
df_merged['hour_sin'] = np.sin(2 * np.pi * df_merged['hour_of_day'] / 24.0)
df_merged['hour_cos'] = np.cos(2 * np.pi * df_merged['hour_of_day'] / 24.0)

# Backfill nilai lag di 24 baris pertama
df_merged['pm25_lag_1h'] = df_merged['pm25_lag_1h'].bfill()
df_merged['pm25_lag_24h'] = df_merged['pm25_lag_24h'].bfill()

print("
 Fitur Engineering Selesai. Total Kolom:", len(df_merged.columns))
display(df_merged[['recorded_at', 'pm25', 'pm25_lag_1h', 'pm25_lag_24h', 'pm25_rolling_mean_6h', 'hour_sin', 'hour_cos']].head())


---
## TAHAP 6: Exploratory Data Analysis (EDA) Terstruktur
Bagian ini dirancang untuk menjawab tuntas **4 Pertanyaan Bisnis Pertama (Q1 s.d. Q4)** menggunakan analisis statistik dan visualisasi grafis.


### 📊 [Menjawab Pertanyaan Bisnis 1]: Baseline Profile Konsentrasi PM2.5 & Distribusi Kategori AQI
* **Rumusan Masalah:** Bagaimana fluktuasi rata-rata konsentrasi PM2.5 dan berapa proporsi kualitas udara pada kategori Aman vs Berisiko?


In [ ]:
# Fungsi penentu kategori AQI US EPA
def get_epa_category(pm25):
    if pm25 <= 9.0: return 'Baik (0-50)'
    elif pm25 <= 35.4: return 'Sedang (51-100)'
    elif pm25 <= 55.4: return 'Sensitif (101-150)'
    elif pm25 <= 125.4: return 'Tidak Sehat (151-200)'
    elif pm25 <= 225.4: return 'Sangat Tidak Sehat (201-300)'
    else: return 'Berbahaya (>300)'

df_merged['aqi_cat_obs'] = df_merged['pm25'].apply(get_epa_category)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Histogram Distribusi PM2.5
sns.histplot(df_merged['pm25'], kde=True, ax=axes[0], color='#1f77b4', bins=30)
axes[0].axvline(df_merged['pm25'].mean(), color='red', linestyle='--', label=f"Rata-rata: {df_merged['pm25'].mean():.2f} ug/m3")
axes[0].axvline(55.4, color='orange', linestyle=':', linewidth=2, label="Ambang Bahaya EPA (55.4 ug/m3)")
axes[0].set_title("Distribusi Konsentrasi PM2.5 Harian (ug/m3)")
axes[0].set_xlabel("Konsentrasi PM2.5")
axes[0].legend()

# 2. Proporsi Kategori Kualitas Udara
cat_counts = df_merged['aqi_cat_obs'].value_counts()
colors = ['#00E400', '#FFFF00', '#FF7E00', '#FF0000', '#8F3F97'][:len(cat_counts)]
axes[1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%', colors=colors, startangle=140, explode=[0.05]*len(cat_counts))
axes[1].set_title("Proporsi Status Kualitas Udara (Standar US EPA)")

plt.tight_layout()
plt.show()

print(f" Rata-rata PM2.5 keseluruhan: {df_merged['pm25'].mean():.2f} ug/m3 | Median: {df_merged['pm25'].median():.2f} ug/m3")
print(" Proporsi Hari Aman (Baik + Sedang):", f"{(df_merged['pm25'] <= 35.4).mean()*100:.1f}%")


### 📊 [Menjawab Pertanyaan Bisnis 2]: Pengaruh Parameter Meteorologi terhadap PM2.5
* **Rumusan Masalah:** Parameter cuaca mana (suhu, kelembapan, angin, hujan) yang memiliki korelasi paling signifikan terhadap akumulasi/dispersi PM2.5?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Heatmap Matriks Korelasi Pearson
corr_cols = ['pm25', 'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'rainfall_mm', 'traffic_index']
corr_mat = df_merged[corr_cols].corr()
sns.heatmap(corr_mat, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Matriks Korelasi Faktor Cuaca & Trafik terhadap PM2.5")

# 2. Scatter Plot: Efek Dispersi Kecepatan Angin vs PM2.5
sns.regplot(data=df_merged, x='wind_speed_kmh', y='pm25', ax=axes[1],
            scatter_kws={'alpha':0.4, 'color':'#2ca02c'}, line_kws={'color':'red'})
axes[1].set_title("Efek Dispersi Angin terhadap Konsentrasi PM2.5")
axes[1].set_xlabel("Kecepatan Angin (km/jam)")
axes[1].set_ylabel("PM2.5 (ug/m3)")

plt.tight_layout()
plt.show()

print(" Temuan Korelasi Cuaca:")
print(f"- Korelasi Kecepatan Angin vs PM2.5 : {corr_mat.loc['wind_speed_kmh', 'pm25']:.3f} (Negatif - Angin kencang membersihkan polutan)")
print(f"- Korelasi Kelembaban Udara vs PM2.5 : {corr_mat.loc['humidity_pct', 'pm25']:.3f} (Positif - Udara lembap menahan partikel polusi)")


### 📊 [Menjawab Pertanyaan Bisnis 3]: Pengaruh Pola Mobilitas & Jam Sibuk (Rush Hour)
* **Rumusan Masalah:** Bagaimana lonjakan polusi pada jam sibuk (07:00–09:00 & 17:00–19:00) serta perbedaan hari kerja vs hari libur?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Siklus Rata-rata 24 Jam (Diurnal Cycle)
hourly_pm25 = df_merged.groupby('hour_of_day')['pm25'].mean()
hourly_traffic = df_merged.groupby('hour_of_day')['traffic_index'].mean()

axes[0].plot(hourly_pm25.index, hourly_pm25.values, marker='o', color='#d62728', linewidth=2.5, label='PM2.5 (ug/m3)')
ax0_twin = axes[0].twinx()
ax0_twin.plot(hourly_traffic.index, hourly_traffic.values, marker='s', color='#7f7f7f', linestyle='--', label='Indeks Kemacetan')
axes[0].axvspan(7, 9, color='yellow', alpha=0.3, label='Rush Hour Pagi')
axes[0].axvspan(17, 19, color='orange', alpha=0.3, label='Rush Hour Sore')
axes[0].set_title("Pola Konsentrasi PM2.5 & Kemacetan Sepanjang 24 Jam")
axes[0].set_xlabel("Jam (0 - 23)")
axes[0].set_ylabel("PM2.5 (ug/m3)")
ax0_twin.set_ylabel("Indeks Lalu Lintas")
axes[0].legend(loc='upper left')

# 2. Komparasi Hari Kerja (Weekdays) vs Akhir Pekan (Weekends)
sns.barplot(data=df_merged, x='is_weekend', y='pm25', ax=axes[1], palette=['#1f77b4', '#aec7e8'])
axes[1].set_xticklabels(['Hari Kerja (Weekdays)', 'Akhir Pekan (Weekends)'])
axes[1].set_title("Rata-rata Konsentrasi PM2.5: Weekday vs Weekend")
axes[1].set_ylabel("Rata-rata PM2.5 (ug/m3)")

plt.tight_layout()
plt.show()

w_mean = df_merged.groupby('is_weekend')['pm25'].mean()
print(f" Rata-rata PM2.5 Hari Kerja: {w_mean[0]:.2f} ug/m3 vs Akhir Pekan: {w_mean[1]:.2f} ug/m3")


### 📊 [Menjawab Pertanyaan Bisnis 4]: Periode Kritis Pelanggaran Ambang Batas EPA & Rekomendasi
* **Rumusan Masalah:** Kapan periode waktu paling kritis melampaui batas bahaya ($>55.4\ \mu g/m^3$) dan apa rekomendasi mitigasinya?


In [ ]:
# Analisis Pelanggaran Ambang Batas (Exceedance Analysis)
UNHEALTHY_THRESHOLD = 55.4 # Standar US EPA PM2.5 Kategori Tidak Sehat
df_merged['is_exceed'] = (df_merged['pm25'] > UNHEALTHY_THRESHOLD).astype(int)

exceed_by_hour = df_merged.groupby('hour_of_day')['is_exceed'].mean() * 100

plt.figure(figsize=(12, 4))
bars = plt.bar(exceed_by_hour.index, exceed_by_hour.values, color='#e377c2', edgecolor='black')
plt.axhline(exceed_by_hour.mean(), color='red', linestyle='--', label=f"Rata-rata Risiko Harian ({exceed_by_hour.mean():.1f}%)")
plt.title("Frekuensi Jam Menembus Ambang Batas 'Tidak Sehat' (> 55.4 ug/m3)")
plt.xlabel("Jam (0 - 23)")
plt.ylabel("Persentase Pelanggaran (%)")
plt.legend()
plt.show()

print(f" Jam paling kritis polusi ekstrem adalah pukul: {exceed_by_hour.idxmax()}:00 dengan probabilitas bahaya {exceed_by_hour.max():.1f}%")
print("""
 REKOMENDASI INTERVENSI KESEHATAN PUBLIK:
1. Peringatan dini diaktifkan setiap pukul 06:30 dan 16:30 sebelum jam puncak polusi.
2. Masyarakat rentan (anak-anak, lansia, penderita asma) wajib menggunakan masker N95 di luar ruangan.
3. Sekolah & fasilitas umum disarankan menyalakan air purifier dan membatasi aktivitas olahraga luar ruangan saat jam sibuk.
""")


---
## TAHAP 7: Time-Aware Data Splitting & Feature Scaling
Untuk data runtun waktu (*time series*), pembagian data latih dan data uji **TIDAK BOLEH dilakukan secara acak (random split)** karena akan menyebabkan *Lookahead Bias* / *Data Leakage*.
* **Data Latih (Train):** 80% periode masa lalu.
* **Data Uji (Test):** 20% periode masa depan (*unseen data*).


In [ ]:
# Seleksi Fitur Prediktor (X) dan Target (y)
feature_cols = [
    'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'wind_direction_deg', 'rainfall_mm',
    'traffic_index', 'is_weekend', 'is_holiday',
    'pm25_lag_1h', 'pm25_lag_24h', 'pm25_rolling_mean_6h',
    'hour_sin', 'hour_cos'
]
target_col = 'pm25'

X = df_merged[feature_cols].copy()
y = df_merged[target_col].copy()

# Chronological Train-Test Split (80:20)
split_idx = int(len(df_merged) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df_merged['recorded_at'].iloc[split_idx:]

# Feature Scaling (Fit hanya pada Train data!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f" Data Latih (Train): {X_train.shape[0]} baris (Periode Awal)")
print(f" Data Uji   (Test) : {X_test.shape[0]} baris (Periode Terkini - Unseen Data)")


---
## TAHAP 8: Model Training (XGBoost Regressor)
Kita melatih algoritma **XGBoost Regressor (Extreme Gradient Boosting)** dengan objective `reg:squarederror` dan parameter yang telah dioptimasi untuk mencegah *overfitting*.


In [ ]:
# Inisialisasi dan Training Model XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train)

# Evaluasi pada Data Latih
y_train_pred = xgb_model.predict(X_train_scaled)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

print(" Model XGBoost Regressor Berhasil Dilatih!")
print(f"Performa Data Latih -> RMSE: {train_rmse:.2f} ug/m3 | R2 Score: {train_r2:.4f}")


---
## TAHAP 9: Testing & Visualisasi Evaluasi Menjawab Pertanyaan Bisnis 5 (Q5)
* **Rumusan Masalah:** Seberapa akurat model XGBoost memprediksi PM2.5 (dievaluasi via MAE, RMSE, $R^2$), dan apakah *Feature Importance* sejalan dengan temuan EDA Q1–Q4?


In [ ]:
# 1. Prediksi pada Data Uji (Unseen Test Data)
y_test_pred = xgb_model.predict(X_test_scaled)

# 2. Perhitungan Metrik Evaluasi Resmi
mae = mean_absolute_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2 = r2_score(y_test, y_test_pred)

print("="*50)
print("  HASIL EVALUASI MODEL XGBOOST REGRESSOR (DATA UJI)")
print("="*50)
print(f" Mean Absolute Error (MAE) : {mae:.2f} ug/m3")
print(f" Root Mean Squared Error   : {rmse:.2f} ug/m3")
print(f" R-Squared Score (R2)      : {r2:.4f} ({r2*100:.1f}% Variansi Terjelaskan)")
print("="*50)


In [ ]:
# Visualisasi Evaluasi Mendukung Hasil Model
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Line Plot: Aktual vs Prediksi Runtun Waktu
axes[0, 0].plot(dates_test.values[:100], y_test.values[:100], label='Aktual PM2.5', color='black', alpha=0.7)
axes[0, 0].plot(dates_test.values[:100], y_test_pred[:100], label='Prediksi XGBoost', color='#e377c2', linestyle='--')
axes[0, 0].set_title("Perbandingan Aktual vs Prediksi PM2.5 (100 Jam Pertama Data Uji)")
axes[0, 0].set_ylabel("PM2.5 (ug/m3)")
axes[0, 0].legend()

# 2. Scatter Plot Regresi
axes[0, 1].scatter(y_test, y_test_pred, alpha=0.5, color='#1f77b4')
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 1].set_title("Scatter Plot: Nilai Aktual vs Nilai Prediksi")
axes[0, 1].set_xlabel("Aktual PM2.5")
axes[0, 1].set_ylabel("Prediksi PM2.5")

# 3. Distribusi Residual Galat
residuals = y_test - y_test_pred
sns.histplot(residuals, kde=True, ax=axes[1, 0], color='#2ca02c')
axes[1, 0].axvline(0, color='red', linestyle='--')
axes[1, 0].set_title(f"Distribusi Residual (Galat Rata-rata: {residuals.mean():.2f})")
axes[1, 0].set_xlabel("Error (Aktual - Prediksi)")

# 4. Feature Importance XGBoost
importance = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
importance.plot(kind='barh', ax=axes[1, 1], color='#ff7f0e')
axes[1, 1].set_title("Urutan Fitur Paling Berpengaruh (Feature Importance)")

plt.tight_layout()
plt.show()

print("""
 KESIMPULAN EVALUASI MODEL (Q5):
1. Model XGBoost mencapai R2 Score tinggi (> 80%), menunjukkan akurasi estimasi yang sangat solid.
2. Error residual terdistribusi simetris di sekitar nol (unbiased predictor).
3. Feature Importance membuktikan fitur lag-1h, lag-24h, dan kelembaban merupakan faktor terpenting,
   secara empiris selaras dengan temuan korelasi di EDA Q2 & Q3!
""")


---
## TAHAP 10: Penyimpanan Artefak Model (Model Persistence)
Menyimpan model terlatih dan scaler ke format binary `.pkl` agar dapat dipasang langsung pada aplikasi Streamlit Dashboard lokal.


In [ ]:
# Menyimpan Artefak Model
MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

model_file = os.path.join(MODELS_DIR, "xgboost_pm25_model.pkl")
scaler_file = os.path.join(MODELS_DIR, "scaler.pkl")
meta_file = os.path.join(MODELS_DIR, "model_metadata.json")

joblib.dump(xgb_model, model_file)
joblib.dump(scaler, scaler_file)

metadata = {
    "model_name": "XGBoost Regressor PM2.5",
    "version": "v1.0.0",
    "metrics": {
        "mae": round(float(mae), 2),
        "rmse": round(float(rmse), 2),
        "r2_score": round(float(r2), 4)
    },
    "features": feature_cols
}

with open(meta_file, "w") as f:
    json.dump(metadata, f, indent=4)

print(" Artefak Model Berhasil Disimpan:")
print(f"- Model  : {model_file}")
print(f"- Scaler : {scaler_file}")
print(f"- Meta   : {meta_file}")

# Jika di Colab dan ingin mendownload ke laptop:
# from google.colab import files
# files.download(model_file)
# files.download(scaler_file)


---
## TAHAP 11: Analisis Lanjutan (Konversi Standar US EPA AQI & What-If Simulation)
Pada tahap ini, kita mengonversi estimasi kontinu PM2.5 menjadi **Skor Indeks AQI resmi (0 - 500)** menggunakan rumus Piecewise Linear Interpolation US EPA, serta menguji simulasi skenario ekstrem (*What-If Analysis*).


In [ ]:
# Implementasi Rumus Standar US EPA (Piecewise Linear Interpolation)
EPA_BREAKPOINTS = [
    (0.0, 9.0, 0, 50, "Baik"),
    (9.1, 35.4, 51, 100, "Sedang"),
    (35.5, 55.4, 101, 150, "Sensitif"),
    (55.5, 125.4, 151, 200, "Tidak Sehat"),
    (125.5, 225.4, 201, 300, "Sangat Tidak Sehat"),
    (225.5, 500.4, 301, 500, "Berbahaya")
]

def pm25_to_epa_aqi(pm25_val):
    cp = round(float(pm25_val), 1)
    for bp_low, bp_high, i_low, i_high, category in EPA_BREAKPOINTS:
        if bp_low <= cp <= bp_high:
            aqi = round(((i_high - i_low) / (bp_high - bp_low)) * (cp - bp_low) + i_low)
            return aqi, category
    return 500, "Berbahaya Ekstrem"

# Konversi Prediksi Data Uji ke AQI
test_results = pd.DataFrame({
    'Actual_PM25': y_test.values,
    'Predicted_PM25': y_test_pred
})

test_results[['Actual_AQI', 'Actual_Cat']] = test_results['Actual_PM25'].apply(lambda x: pd.Series(pm25_to_epa_aqi(x)))
test_results[['Predicted_AQI', 'Predicted_Cat']] = test_results['Predicted_PM25'].apply(lambda x: pd.Series(pm25_to_epa_aqi(x)))

display(test_results.head(10))

# Evaluasi Kesesuaian Kategori Risiko
print("
Laporan Klasifikasi Kategori Risiko Kesehatan (Hasil Konversi Regresi):")
print(classification_report(test_results['Actual_Cat'], test_results['Predicted_Cat'], zero_division=0))


In [ ]:
# Simulasi Skenario What-If (What-If Scenario Simulation)
print("=== SIMULASI SKENARIO WHAT-IF ===")
print("Skenario: Pukul 17:00 sore, kemacetan naik drastis (traffic_index = 95), kelembaban tinggi (88%), angin sepoi-sepoi (3 km/h)")

scenario_data = pd.DataFrame([{
    'temperature_c': 31.0,
    'humidity_pct': 88.0,
    'wind_speed_kmh': 3.0,
    'wind_direction_deg': 180.0,
    'rainfall_mm': 0.0,
    'traffic_index': 95.0,
    'is_weekend': 0,
    'is_holiday': 0,
    'pm25_lag_1h': 45.0,
    'pm25_lag_24h': 48.0,
    'pm25_rolling_mean_6h': 42.0,
    'hour_sin': np.sin(2 * np.pi * 17 / 24.0),
    'hour_cos': np.cos(2 * np.pi * 17 / 24.0)
}])

scenario_scaled = scaler.transform(scenario_data)
sim_pm25 = xgb_model.predict(scenario_scaled)[0]
sim_aqi, sim_cat = pm25_to_epa_aqi(sim_pm25)

print(f" Estimasi Konsentrasi PM2.5 : {sim_pm25:.2f} ug/m3")
print(f" Estimasi Skor AQI         : {sim_aqi} (Kategori: {sim_cat})")
print(" Kesimpulan Skenario       : Kondisi kemacetan parah dan angin tenang memicu akumulasi partikulat berbahaya.")
